In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
import os


In [ ]:
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')
model_name = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash')

model = ChatGoogleGenerativeAI(
    model=model_name,
    api_key=api_key,
)


In [ ]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str


In [ ]:
def generate_joke(state: JokeState) -> dict:
    prompt = f"Generate a joke on the topic: {state['topic']}"
    response = model.invoke(prompt).content
    return {'joke': response}


In [ ]:
def generate_explanation(state: JokeState) -> dict:
    prompt = f"Write a short explanation for the following joke:

{state['joke']}"
    response = model.invoke(prompt).content
    return {'explanation': response}


In [ ]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)


In [ ]:
config1 = {'configurable': {'thread_id': '1'}}
workflow.invoke({'topic': 'pizza'}, config=config1)


In [ ]:
workflow.get_state(config1)


In [ ]:
list(workflow.get_state_history(config1))
